# 추세 차트 — 선 그래프 · 다중 계열 · 이동평균


**설명 전용 자료:** 개념·예제·주의점을 Markdown 셀로 제공합니다. 연습문제와 빈 풀이 셀은 같은 이름의 `_Practice.ipynb`에 있습니다.

**학습 목표:** 시계열 및 순서형 데이터를 `plt.plot()`과 `fig, ax = plt.subplots()`로 시각화하고, 다중 선 스타일·범례·구간 확대·서브플롯과 이동평균(`rolling(window=N).mean()`)으로 장기 추세를 파악합니다.

## 처음 배우는 분을 위한 안내

선 그래프는 순서가 있는 값의 변화를 연결합니다. 날짜형 변환 후 시간순으로 정렬해야 합니다. 이동평균은 현재 행과 앞의 여러 행을 묶은 평균입니다. window=3은 3개 관측값이며, 매일 한 행일 때 3일을 뜻합니다. 누락 날짜가 있으면 먼저 날짜 간격을 확인합니다.

**학습 순서:** 문제 1~3은 필수 기초, 4~12는 선택 도전입니다. 후반 문제는 1번의 online_daily와 3번의 월별 자료를 사용하므로 순서대로 진행합니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

그래프 창이 비어 있으면 데이터 개수와 결측 제거 결과를 먼저 확인하세요. 한글이 네모로 나오면 글꼴 설정 문제이며 계산 오류와 구분합니다.

## 실행 환경과 예제 데이터

아래 가상 데이터를 공통으로 사용합니다. 코드 블록은 읽기용이며 실행하려면 별도 코드 셀에 복사하세요.

```python
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 날짜 순서가 일부 섞여 있는 일별 온라인·오프라인 가상 데이터 (10일)
online_daily_raw = pd.DataFrame({
    'order_date': [
        '2026-01-03', '2026-01-01', '2026-01-02', '2026-01-05', '2026-01-04',
        '2026-01-06', '2026-01-08', '2026-01-07', '2026-01-09', '2026-01-10'
    ],
    'orders': [9, 10, 13, 11, 14, 15, 16, 12, 18, 20],
    'revenue': [90, 100, 130, 110, 140, 150, 160, 120, 180, 210]
})

offline_daily_raw = pd.DataFrame({
    'order_date': pd.date_range('2026-01-01', periods=10).strftime('%Y-%m-%d'),
    'orders': [6, 8, 5, 9, 7, 10, 8, 11, 9, 12],
    'revenue': [70, 85, 65, 95, 80, 110, 90, 125, 105, 140]
})

# 월별·카테고리별 매출 가상 데이터 (1~6월)
monthly_cat = pd.DataFrame({
    'order_date': [
        '2026-01-15', '2026-01-20', '2026-02-10', '2026-02-18',
        '2026-03-05', '2026-03-22', '2026-04-11', '2026-04-25',
        '2026-05-09', '2026-05-19', '2026-06-14', '2026-06-28'
    ],
    'product_category': ['Tech', 'Food'] * 6,
    'revenue': [300, 180, 340, 195, 320, 210, 390, 225, 430, 240, 480, 260]
})

# 실제값 vs 예측값 비교용 데이터
eval_df = pd.DataFrame({
    'step': range(1, 11),
    'actual': [100, 256, 152, 140, 150, 183, 175, 203, 152, 198],
    'predicted': [115, 242, 176, 148, 115, 209, 185, 256, 129, 195]
})
```

## 선 그래프를 그리기 전 필수 점검: 날짜형 변환과 시간순 정렬

선 그래프(`plt.plot`)는 데이터프레임에 들어 있는 **행 순서 그대로** 점들을 이어 그립니다. 만약 날짜가 문자열이거나 순서가 섞여 있으면 선이 앞뒤로 지그재그로 엉키게 됩니다.

1. `pd.to_datetime()`으로 날짜 열을 `datetime64` 자료형으로 변환합니다.
2. 일별/월별 집계를 수행한 뒤 반드시 날짜(또는 시간 인덱스) 기준으로 오름차순 정렬(`sort_values` 또는 `sort_index`)합니다.

| 방식 | 코드 패턴 | 특징과 권장 상황 |
| --- | --- | --- |
| 함수형 (`pyplot`) | `plt.plot(x, y)` → `plt.xlabel()` → `plt.title()` | 단일 차트를 빠르게 확인할 때 간편함 |
| 객체지향 (`OOP`) | `fig, ax = plt.subplots()` → `ax.plot()` → `ax.set_title()` | 여러 차트(`subplots`)를 제어하거나 세부 속성을 정밀하게 다룰 때 표준 방식 |

## 선 스타일과 차트 완성도 속성 (축 · 제목 · 범례 · 격자)

하나의 차트에 두 개 이상의 선을 그릴 때는 색상뿐만 아니라 **선 모양(`linestyle`)과 마커(`marker`)**를 다르게 지정해야 흑백 인쇄나 색각 이상 환경에서도 명확히 구분됩니다.

| 옵션 | 주요 설정값 | 설명 |
| --- | --- | --- |
| `color` | `'steelblue'`, `'coral'`, `'green'`, `'#FF5733'` | 선과 마커의 색상 |
| `linestyle` | `'-'`(실선), `'--'`(파선), `':'`(점선), `'-.'`(일점쇄선) | 선의 스타일 |
| `marker` | `'o'`(원), `'s'`(사각형), `'^'`(삼각형), `'v'`, `'D'` | 데이터 포인트 위치 표시 기호 |
| `linewidth` / `alpha` | `2.0` / `0.0 ~ 1.0` | 선 두께 / 투명도 (원본 노이즈 선은 `alpha=0.4`로 흐리게 처리) |

```python
online_sorted = online_daily_raw.assign(
    order_date=pd.to_datetime(online_daily_raw['order_date'])
).sort_values('order_date')
offline_sorted = offline_daily_raw.assign(
    order_date=pd.to_datetime(offline_daily_raw['order_date'])
).sort_values('order_date')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(online_sorted['order_date'], online_sorted['revenue'],
        marker='o', linestyle='-', color='steelblue', label='Online')
ax.plot(offline_sorted['order_date'], offline_sorted['revenue'],
        marker='s', linestyle='--', color='coral', label='Offline')
ax.set_xlabel('날짜')
ax.set_ylabel('매출')
ax.set_title('채널별 일별 매출 추세 비교')
ax.legend(title='채널')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
```

## 구간 확대(`xlim`, `ylim`), 주석(`annotate`), 다중 하위 차트(`subplot`)

- **축 범위 지정(`plt.xlim`, `plt.ylim` 또는 `ax.set_xlim`, `ax.set_ylim`)**: 관심 있는 기간이나 값 구간을 확대하거나, 두 그래프의 Y축 스케일을 통일해 착시를 방지할 때 사용합니다.
- **데이터 강조(`plt.annotate`)**: 최고점·급락 시점 등 주요 이벤트 위치(`xy`)에 화살표(`arrowprops`)와 설명 텍스트(`xytext`)를 표시합니다.
- **하위 차트(`plt.subplot(행, 열, 순서)` 또는 `plt.subplots(행, 열)`)**: 여러 지표를 한 화면의 대시보드 형태로 배치할 때 사용합니다.

```python
eval_df['error'] = eval_df['actual'] - eval_df['predicted']
plt.figure(figsize=(7, 4))
plt.plot(eval_df['step'], eval_df['actual'], 'g-o', label='실제값(actual)')
plt.plot(eval_df['step'], eval_df['predicted'], 'r--s', label='예측값(predicted)')
plt.plot(eval_df['step'], eval_df['error'], 'b:^', label='오차(error)')
plt.annotate('최대 실제값(256)', xy=(2, 256), xytext=(3.2, 265),
             arrowprops={'color': 'green', 'shrink': 0.05})
plt.ylim(-80, 300)
plt.xlabel('구간(step)')
plt.ylabel('값')
plt.title('실제값 · 예측값 · 오차 추이')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
```

## 이동평균(`rolling(window=N).mean()`)의 원리와 `window` 선택

일별 매출이나 주문량은 요일 효과나 일시적 이벤트 때문에 매일 들쭉날쭉(노이즈)합니다. **이동평균(Rolling Average)**은 현재 시점을 포함해 직전 `N`개 데이터의 평균을 연속적으로 계산하여 단기 변동을 평탄화하고 전체 추세를 부각합니다.

- **초기 `NaN` 발생 이유:** `window=3`이면 첫 번째와 두 번째 날짜에는 평균을 낼 3개의 데이터가 아직 모이지 않았으므로 결과 앞부분 `N-1`개(`2`개) 값이 `NaN`이 되며, 선은 3번째 날짜부터 그려집니다.
- **단기 vs 장기 `window`:** `window=3`처럼 작으면 최근 변화를 빠르게 반영하지만 흔들림이 남고, `window=7` 이상으로 크면 곡선이 훨씬 매끄러워지지만 급격한 변화 시점이 뒤늦게 반영됩니다.

```python
online_sorted['ma3'] = online_sorted['revenue'].rolling(window=3).mean()
online_sorted['ma5'] = online_sorted['revenue'].rolling(window=5).mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(online_sorted['order_date'], online_sorted['revenue'],
        color='steelblue', alpha=0.4, marker='o', label='일별 매출(원본)')
ax.plot(online_sorted['order_date'], online_sorted['ma3'],
        color='coral', linewidth=2, label='3일 이동평균')
ax.plot(online_sorted['order_date'], online_sorted['ma5'],
        color='seagreen', linestyle='--', linewidth=2, label='5일 이동평균')
ax.set_title('일별 매출과 이동평균(3일·5일) 비교')
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
```

## 선택 도전: 같은 날짜 주문을 먼저 합치기

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_orders = pd.DataFrame({'date':['2026-01-01','2026-01-01','2026-01-02'],'revenue':[10,20,40]})
```

### 진행 순서

1. date를 pd.to_datetime으로 날짜형으로 바꿉니다.
2. 날짜별 revenue 합계를 구하고 날짜순으로 정렬합니다.
3. 일별 합계로 선 그래프를 그립니다. 주문별 3행을 바로 연결한 경우와 무엇이 다른지 적습니다.

**완료 기준:** 1월1일 30, 1월2일 40의 두 점. 일별 합계 70은 주문 합계 70과 같음.

### 선택 도전 풀이

```python
import matplotlib.pyplot as plt
extra_orders['date']=pd.to_datetime(extra_orders['date'])
extra_daily=extra_orders.groupby('date')['revenue'].sum().sort_index()
print(extra_daily)
plt.plot(extra_daily.index,extra_daily.values,marker='o')
plt.xlabel('Date');plt.ylabel('Revenue');plt.title('Daily Total');plt.show()
assert extra_daily.tolist()==[30,40] and extra_daily.sum()==70
```

## 복습 순서

1. 날짜 열을 `pd.to_datetime()`으로 변환하고 시간순으로 정렬했는지 먼저 확인합니다.
2. 단일 선 그래프와 다중 선 그래프에서 `marker`, `linestyle`, `legend`, `grid`를 설정합니다.
3. `rolling(window=N).mean()` 실행 후 앞의 `N-1`개 행이 `NaN`이 되는 이유를 수식으로 확인합니다.
4. 원본 선(`alpha=0.4`)과 이동평균 선(`linewidth=2`)을 겹쳐 그려 단기 변동과 장기 추세를 함께 해석합니다.